# Finite Difference Methods for Option Pricing

This notebook develops and implements finite-difference schemes for solving the transformed Black–Scholes PDE in order to price European call options. The focus is on translating the continuous pricing problem into a stable and efficient numerical framework.

We work in log-price space and backward time, leading to a constant-coefficient PDE that is well-suited for grid-based discretization.

We consider:

- Explicit finite-difference schemes (FTCS) for forward time stepping.
- Implicit Crank–Nicolson schemes for improved stability and accuracy.
- Matrix-based formulations of the discretized system using tri-diagonal operators.

Throughout, we emphasize the stability, convergence, and implementation efficiency of such PDE solvers.

### Structure

1. **Grid construction and transformations**
   - Log-price domain truncation and discretization.
   - Time-stepping framework for backward PDE solving.

2. **FTCS scheme implementation**
   - Explicit update rule.
   - Stability considerations and parameter constraints.

3. **Crank–Nicolson scheme**
   - Semi-implicit formulation.
   - Construction of tri-diagonal matrices.
   - Linear system solution at each time step.

4. **Boundary and terminal conditions**
   - Incorporation of payoff at maturity.
   - Treatment of truncated spatial boundaries.

5. **Numerical experiments**
   - Pricing European call options across moneyness regimes.
   - Comparison between FTCS and Crank–Nicolson results.

6. **Convergence and mesh analysis**
   - Empirical verification of second-order accuracy.
   - Sensitivity to grid resolution and time step.

7. **Delta estimation**
   - Numerical differentiation in the original price variable.
   - Interpretation of the resulting sensitivity profiles.


## 1. Grid Construction and Transformations

We transform the Black–Scholes PDE into log-space:

$$
X = \log(S), \quad \tau = T - t
$$

The transformed PDE becomes:

$$
\frac{\partial V}{\partial \tau}
=
\left(r - \frac{1}{2}\sigma^2\right)\frac{\partial V}{\partial X}
+
\frac{1}{2}\sigma^2 \frac{\partial^2 V}{\partial X^2}
-
rV
$$

an initial bounded value problem (IBVP) with constant coefficients.

We define a uniform grid:

- Space: $$ X_i = X_{\min} + i \Delta X $$
- Time: $$ \tau_n = (n+1) \Delta \tau $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Parameters
r = 0.04
sigma = 0.30
T = 1.0
K = 110

# Grid parameters
N = 100          # number of interior points
M = 100          # time steps

# Domain
X_min, X_max = -5, 10

dx = (X_max - X_min) / (N + 1)
dt = T / M

# Full grid (including boundaries)
X = np.linspace(X_min, X_max, N + 2)

# Interior points only (unknowns)
X_inner = X[1:-1]

# Corresponding S values
S_inner = np.exp(X_inner)
S_min = np.exp(X_min)
S_max = np.exp(X_max)

## 2. FTCS Scheme Implementation

We define the coefficients

$$
\nu = \frac{\sigma^2}{2}, \quad
\mu = r - \frac{\sigma^2}{2}
$$

The FTCS (Forward Time Centered Space) scheme is given by

$$
V_i^{n+1}
=
V_i^n
+
\frac{\mu \Delta \tau}{2 \Delta X} \left( V_{i+1}^n - V_{i-1}^n \right)
+
\frac{\nu \Delta \tau}{\Delta X^2} \left( V_{i+1}^n - 2V_i^n + V_{i-1}^n \right)
-
r \Delta \tau V_i^n
$$

Rewriting in tri-diagonal form

$$
V_i^{n+1}
=
a_{-1} V_{i-1}^n + a_0 V_i^n + a_1 V_{i+1}^n
$$

with

$$
a_{-1} = -\frac{\mu \Delta \tau}{2 \Delta X} + \frac{\nu \Delta \tau}{\Delta X^2}
$$

$$
a_0 = 1 - \frac{2 \nu \Delta \tau}{\Delta X^2} - r \Delta \tau
$$

$$
a_1 = \frac{\mu \Delta \tau}{2 \Delta X} + \frac{\nu \Delta \tau}{\Delta X^2}
$$

This leads to the matrix formulation

$$
\mathbf{V}^{n+1} = A \mathbf{V}^n
$$

where $A$ is a tri-diagonal matrix ($B=I$).

In [2]:
def build_ftcs_matrix(N, dx, dt, r, sigma):
    nu = 0.5 * sigma**2
    mu = r - 0.5 * sigma**2

    a_m1 = -mu * dt / (2 * dx) + nu * dt / dx**2
    a_0  = 1 - 2 * nu * dt / dx**2 - r * dt
    a_p1 =  mu * dt / (2 * dx) + nu * dt / dx**2

    A = (
        np.diag([a_0] * N)
        + np.diag([a_p1] * (N - 1), 1)
        + np.diag([a_m1] * (N - 1), -1)
    )

    return A, a_m1, a_0, a_p1

## 3. Crank-Nicolson Scheme

The Crank–Nicolson scheme is obtained by averaging the explicit and implicit discretizations

$$
B \mathbf{V}^{n+1} = A \mathbf{V}^n
$$

We define $\alpha$ and $\beta$ in the following way

$$
\alpha = \frac{\nu \Delta \tau}{\Delta X^2}, \qquad
\beta = \frac{\mu \Delta \tau}{2 \Delta X}
$$


For matrix $A$ (explicit part)

$$
a_{-1} = \frac{1}{2} (\alpha - \beta)
$$

$$
a_0 = 1 - \alpha - \frac{r \Delta \tau}{2}
$$

$$
a_1 = \frac{1}{2} (\alpha + \beta)
$$

For matrix $B$ (implicit part)

$$
b_{-1} = -\frac{1}{2} (\alpha - \beta)
$$

$$
b_0 = 1 + \alpha + \frac{r \Delta \tau}{2}
$$

$$
b_1 = -\frac{1}{2} (\alpha + \beta)
$$

In [3]:
def build_crank_nicolson_matrices(N, dx, dt, r, sigma):
    nu = 0.5 * sigma**2
    mu = r - 0.5 * sigma**2

    alpha = nu * dt / dx**2
    beta  = mu * dt / (2 * dx)

    # Matrix A (explicit part)
    a_m1 = 0.5 * (alpha - beta)
    a_0  = 1 - alpha - 0.5 * r * dt
    a_p1 = 0.5 * (alpha + beta)

    A = (
        np.diag([a_0] * N)
        + np.diag([a_p1] * (N - 1), 1)
        + np.diag([a_m1] * (N - 1), -1)
    )

    # Matrix B (implicit part)
    b_m1 = -0.5 * (alpha - beta)
    b_0  = 1 + alpha + 0.5 * r * dt
    b_p1 = -0.5 * (alpha + beta)

    B = (
        np.diag([b_0] * N)
        + np.diag([b_p1] * (N - 1), 1)
        + np.diag([b_m1] * (N - 1), -1)
    )

    return A, B, a_m1, a_0, a_p1, b_m1, b_0, b_p1

## 4. Boundary and Terminal Conditions

The terminal payoff for a European call option is

$$
V(S, T) = \max(S - K, 0)
$$

Using the transformation:

$$
S = e^X
$$

we initialize:

$$
V(X, 0) = \max(e^X - K, 0)
$$

### Boundary conditions

As $S \to 0$:

$$
V \to 0
$$

As $S \to \infty$:

$$
V \sim S - K e^{-r\tau}
$$

In [16]:
def terminal_payoff(X_inner, K):
    S = np.exp(X_inner)
    return np.maximum(S - K, 0)


def boundary_conditions(S_max, K, r, tau):   #intentionally leaving S_min because it approximates the behavior at S=0, which is 0 for a call option
    left = 0.0
    right = S_max - K * np.exp(-r * tau)
    return left, right

For $i = 1$ and $i = N$, the terms $V_0^n$ and $V_{N+1}^n$ are determined by the boundary conditions. These boundary values are incorporated by adding their contributions to the first and last entries of the solution vector at each time step.

This incorporation of boundary conditions will be applied in the implementation of the FTCS and Crank–Nicolson schemes in the next section, where numerical experiments are performed to compute and analyze option prices.